# Demo Topik 5 — Spark Structured Streaming
**Pendamping seluruh slide DEMO pada Deck 05: slide 10, 17, 20, 28, 29, dan 35**

Jalankan cell secara berurutan. Demo memakai Spark `local[2]` agar rate/window tidak menunggu resource cluster yang mungkin sedang dipakai peserta. Kafka tetap diakses melalui `kafka:9092`.

## Setup
Buat satu SparkSession untuk seluruh rangkaian demo.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from datetime import datetime, timezone
import json, time

spark = (
    SparkSession.builder
    .appName("spark-training-deck-05-demos")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

## Slide 10 — Streaming paling sederhana: rate source
Amati bahwa setiap micro-batch hanya berisi event baru. Belum ada window, state, watermark, atau Kafka.

In [ ]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)
out = rate_df.select("timestamp", "value")

q_rate = (
    out.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .queryName("slide10_rate_demo")
    .trigger(processingTime="3 seconds")
    .start()
)
try:
    q_rate.awaitTermination(8)
finally:
    q_rate.stop()
print("Query active after stop:", q_rate.isActive)

## Slide 17 — Window aggregation dengan event time
`timestamp` dari rate source menjadi `event_time`. Tunjukkan struct `window.start/window.end` serta nilai agregasi yang bertambah dalam window yang sama.

In [ ]:
events = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .select(
        F.col("timestamp").alias("event_time"),
        ((F.col("value") % 5 + 1) * 100000).cast("long").alias("amount"),
    )
)
counts = (
    events
    .groupBy(F.window(F.col("event_time"), "1 minute"))
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
)

q_window = (
    counts.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .queryName("slide17_window_demo")
    .trigger(processingTime="5 seconds")
    .start()
)
try:
    q_window.awaitTermination(15)
finally:
    q_window.stop()
print("Query active after stop:", q_window.isActive)

## Slide 20 — Window + watermark
Agar hasil `append` terlihat saat demo, skala waktu dipadatkan dari menit menjadi window 10 detik dan watermark 5 detik. Urutan operator tetap sama: `withWatermark` → `window` → agregasi → sink.

In [ ]:
watermark_events = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .select(
        F.col("timestamp").alias("event_time"),
        F.when(F.col("value") % 2 == 0, "JKT").otherwise("SBY").alias("region"),
        ((F.col("value") % 5 + 1) * 100000).cast("long").alias("amount"),
    )
)
watermarked = (
    watermark_events
    .withWatermark("event_time", "5 seconds")
    .groupBy(F.window("event_time", "10 seconds"), F.col("region"))
    .agg(
        F.count("*").alias("n"),
        F.sum("amount").alias("total_amount"),
    )
)

q_watermark = (
    watermarked.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .queryName("slide20_watermark_demo")
    .trigger(processingTime="3 seconds")
    .start()
)
try:
    q_watermark.awaitTermination(22)
finally:
    q_watermark.stop()
print("Query active after stop:", q_watermark.isActive)

## Setup demo Kafka
Topic `transactions` disiapkan oleh Compose. Helper berikut mengirim event contoh setelah streaming query aktif sehingga `startingOffsets=latest` tetap dapat diamati.

In [ ]:
kafka_bootstrap = "kafka:9092"

def send_kafka_events(rows):
    payload = [
        (
            trx_id,
            json.dumps({
                "trx_id": trx_id,
                "region": region,
                "amount": amount,
                "event_time": event_time.isoformat(),
            }),
        )
        for trx_id, region, amount, event_time in rows
    ]
    (
        spark.createDataFrame(payload, ["key", "value"])
        .selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value")
        .write.format("kafka")
        .option("kafka.bootstrap.servers", kafka_bootstrap)
        .option("topic", "transactions")
        .save()
    )

## Slide 28 — Membaca topic Kafka
Amati bahwa `key` dan `value` awalnya binary, lalu di-cast menjadi string. Pertahankan `topic`, `partition`, `offset`, dan `timestamp` untuk observability.

In [ ]:
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", "transactions")
    .option("startingOffsets", "latest")
    .load()
)
text = raw.selectExpr(
    "CAST(key AS STRING) AS key",
    "CAST(value AS STRING) AS json",
    "topic", "partition", "offset", "timestamp",
)
q_kafka_raw = (
    text.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .queryName("slide28_kafka_raw_demo")
    .trigger(processingTime="2 seconds")
    .start()
)
time.sleep(2)
now = datetime.now(timezone.utc)
send_kafka_events([
    ("D2801", "JKT", 125000, now),
    ("D2802", "SBY", 250000, now),
])
try:
    q_kafka_raw.awaitTermination(8)
finally:
    q_kafka_raw.stop()
print("Query active after stop:", q_kafka_raw.isActive)

## Slide 29 — Parsing JSON dengan schema bisnis
Gunakan schema eksplisit, pertahankan metadata Kafka, dan gunakan `DecimalType` untuk nilai finansial.

In [ ]:
schema = T.StructType([
    T.StructField("trx_id", T.StringType()),
    T.StructField("region", T.StringType()),
    T.StructField("amount", T.DecimalType(14, 2)),
    T.StructField("event_time", T.TimestampType()),
])
parsed = (
    text
    .withColumn("data", F.from_json("json", schema))
    .select("data.*", "partition", "offset")
    .filter(F.col("trx_id").isNotNull())
)
q_kafka_parsed = (
    parsed.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .queryName("slide29_kafka_parsed_demo")
    .trigger(processingTime="2 seconds")
    .start()
)
time.sleep(2)
now = datetime.now(timezone.utc)
send_kafka_events([
    ("D2901", "JKT", 375000, now),
    ("D2902", "BDG", 500000, now),
])
try:
    q_kafka_parsed.awaitTermination(8)
finally:
    q_kafka_parsed.stop()
print("Query active after stop:", q_kafka_parsed.isActive)

## Slide 35 — Membaca progress secara interaktif
Mulai ulang window query ke memory sink, lalu baca `lastProgress`. Bandingkan input rate, processed rate, durasi trigger, sources, dan state operators.

In [ ]:
q_progress = (
    counts.writeStream
    .format("memory")
    .queryName("slide35_progress_demo")
    .outputMode("update")
    .trigger(processingTime="3 seconds")
    .start()
)
try:
    q_progress.awaitTermination(10)
    progress = q_progress.lastProgress
    print(json.dumps(progress, indent=2))
    if progress:
        print("input r/s:", progress.get("inputRowsPerSecond"))
        print("processed r/s:", progress.get("processedRowsPerSecond"))
        print("duration:", progress.get("durationMs"))
        print("sources:", progress.get("sources"))
        print("state:", progress.get("stateOperators"))
finally:
    q_progress.stop()
print("Query active after stop:", q_progress.isActive)

## Cleanup
Hentikan query yang masih aktif dan tutup SparkSession sebelum kembali ke notebook lab.

In [ ]:
for query in spark.streams.active:
    print("Stopping:", query.name)
    query.stop()
print("Active queries:", len(spark.streams.active))
spark.stop()
print("SparkSession stopped")